In [1]:
"""
Exp01D: Block Bootstrap PnL Distribution (Path Uncertainty)

Goal:
- Treat backtest outcomes (Sharpe / Annual Return / MDD) as random variables under path-level uncertainty (return ordering / clustering).
- Produce an empirical distribution of strategy performance by resampling the full-sample strategy return path via block bootstrap.

Setting:
- Daily data
- Long-only, single MA (already in your pipeline)
- Signal-only baseline (no vol targeting / no risk-off)
- COST_RATE typically set to 0 here (cost handled separately in Exp01A)

Bootstrap Setup:
- Base series: strategy daily returns ret_t from full-sample backtest


'\nExp01C: Metric Sampling Distribution (Rolling Windows)\n\nGoal:\n- Treat backtest metrics (Sharpe / Annual Return / MDD / Turnover) as random variables\n  under finite-sample uncertainty.\n- Produce an empirical distribution over rolling sub-samples (windows).\n\nSetting:\n- Daily data\n- Long-only, single MA (already in your pipeline)\n- Signal-only baseline (no vol targeting / no risk-off)\n- COST_RATE typically set to 0 here (you already handled cost in Exp01A)\n\nMethod:\n- Build rolling windows over the available trading dates\n- For each window: run(cfg with START/END), collect summary metrics\n- Analyze the distribution: quantiles + tail probabilities\n'

In [2]:
import os
os.chdir("..")
print(os.getcwd())

/Users/kim/Desktop/Quant-MA


In [3]:
from copy import deepcopy
import pandas as pd

from config import Config
from runner import run

from dataclasses import replace
cfg_base = Config()
cfg_base = replace(cfg_base, MA_WINDOW=80, COST_RATE=0)   # 选一个 Exp01 plateau 的中心80, COST_RATE=0

In [4]:
# ----------------
# ✅ Cell 1：固定 full sample 跑一次 baseline
# ----------------

import numpy as np
import pandas as pd
from dataclasses import replace

# full sample（和你 Exp01B/C 一致）
cfg_full = replace(cfg_base, START="2015-01-02", END="2025-01-03")

bt_full, summary_full = run(cfg_full)

print("summary_full:", summary_full)
print("bt_full columns:", bt_full.columns.tolist()[:30])  # 先看前30列
bt_full.head()


summary_full: {'Annual Return': 0.08182218543831343, 'Max Drawdown': -0.16943727065147374, 'Sharpe': 0.7653138601951989, 'Total Turnover': 108.0, 'N_obs': 2436}
bt_full columns: ['price', 'price_ret', 'signal', 'position', 'turnover', 'ret', 'equity']


/Users/kim/Desktop/Quant-MA/data/loaders.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(path, index_col=0, parse_dates=True)
/Users/kim/Desktop/Quant-MA/data/loaders.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  idx = pd.to_datetime(df.index, errors="coerce")


,price,price_ret,signal,position,turnover,ret,equity
Price,,,,,,,
2015-01-02,170.589630,NaN,NaN,NaN,NaN,NaN,1.0
2015-01-05,167.508881,-0.018059,NaN,NaN,0.0,NaN,1.0
2015-01-06,165.931076,-0.009419,NaN,NaN,0.0,NaN,1.0
2015-01-07,167.998718,0.012461,NaN,NaN,0.0,NaN,1.0
2015-01-08,170.979904,0.017745,NaN,NaN,0.0,NaN,1.0


In [5]:
# ✅ Cell 2：确定“策略日收益”的列名（非常关键）

candidates = [c for c in bt_full.columns if "ret" in c.lower() or "pnl" in c.lower() or "equity" in c.lower()]
candidates

'''
Step 2｜确认“用于 bootstrap 的收益定义”（现在这一步）

1️⃣ price_ret
含义：标的资产本身的日收益
❌ 不是策略收益
❌ 用它做 bootstrap = 在问“SPY 本身稳不稳”
👉 Exp00 / benchmark 才用
结论：❌ 不用


2️⃣ equity
含义：策略累计净值曲线
不是收益，是 level
bootstrap 不能直接对 level 做（会破坏动态）
如果要用，必须先 equity.pct_change() 或 log diff
结论：⚠️ 可推导，但不是最干净

3️⃣ ret
含义：策略的日收益（已经过 position / execution）
正是你 Exp01A/B/C 算 Sharpe、Annual Return 的那个对象
和你现在 summary 完全一致
结论：✅ 唯一正确选择
'''

['price_ret', 'ret', 'equity']

In [6]:
# ✅ Step 3: extract strategy daily returns for bootstrap

ret = bt_full["ret"].dropna().values

len(ret), ret[:5], ret[-5:]


(2436,
 array([-0.00411457, -0.0100203 ,  0.01084131,  0.0028476 , -0.01145205]),
 array([ 1.11150097e-02,  6.65479295e-05, -1.05264942e-02, -1.14116632e-02,
        -3.63811544e-03]))

In [7]:
# ✅ Step 4: block bootstrap 抽样，构造path
'''
你在抽什么“路径”？
你有一条真实历史的策略日收益序列：ret[0], ret[1], ..., ret[T-1], 你的 T = 2436（2015–2025 日频）
这条序列就是一条历史路径（path）：市场按这个顺序走，你的策略就按这个顺序赚/亏。

Exp01D 的问题是：如果“同样这些日收益的局部结构”存在，但时间顺序不同（不同路径），策略指标会不会崩？
所以你做的是：
✅ 不改变单日收益的数值范围（还是来自历史）
✅ 不做任何模型假设（不拟合分布）
✅ 只通过“重排”构造很多条可能的路径

为什么不是随便打乱（shuffle）？
如果你直接 i.i.d. shuffle（每一天随机抽一日收益）：会破坏收益序列的 自相关/聚类
日频里常见现象：波动聚类、趋势持续、回撤持续（你 MA 信号尤其依赖这种结构）
那样的 bootstrap 往往会过于乐观。
所以你用了 block bootstrap：以“块”为单位重排，尽量保留局部时间结构。

'''


import numpy as np

# Step 4.1｜写 block bootstrap 函数（核心但很短）
def block_bootstrap_returns(returns, block_size, n_samples):
    """
    Block bootstrap for time-series returns.
    
    returns: 1D numpy array (strategy daily returns)
    block_size: number of consecutive days per block
    n_samples: number of bootstrap paths
    """
    T = len(returns)
    n_blocks = int(np.ceil(T / block_size)) # e.g., T/block_size = 2436/21 ≈ 116 --> 每条 bootstrap path 都会由 116 个“21天连续收益块”拼起来，拼完再裁剪到 2436 天

    paths = []
    for _ in range(n_samples):  # 重复 1000 次得到分布
        idx = np.random.randint(0, T - block_size + 1, size=n_blocks)  # 随机抽“块的起点”
        boot = np.concatenate([returns[i:i + block_size] for i in idx])[:T] 
        # 每个数字 i 表示：从原始序列里取一个连续块：returns[i : i+block_size]; 
        # NB：抽样是 with replacement（可重复抽到同一个块），这才叫 bootstrap
        # boot = np.concatenate([returns[i:i + block_size] for i in idx])[:T]: 把抽到的块拼接成一条新路径,如果拼出来略长，就 [:T] 裁掉
        # 终得到：boot.shape == (2436,), 这就是一条 bootstrap path。
        paths.append(boot)

    return paths

# Step 4.2｜确定 bootstrap 的“实验参数”（写进 Method）
BLOCK_SIZE = 21     # ~1 trading month
N_BOOT     = 1000   # number of bootstrap paths

# Step 4.3｜生成 bootstrap 路径（不算指标）
boot_paths = block_bootstrap_returns(
    returns=ret,
    block_size=BLOCK_SIZE,
    n_samples=N_BOOT
)

len(boot_paths), boot_paths[0][:5]


(1000, array([0.00124678, 0.00477783, 0.0023444 , 0.00447699, 0.00033269]))

In [8]:
# ✅ Step 5｜对每条 bootstrap 路径计算指标（核心结果）

TRADING_DAYS = 252

def max_drawdown_from_returns(r):
    equity = np.cumprod(1 + r)
    peak = np.maximum.accumulate(equity)
    dd = equity / peak - 1.0
    return dd.min()

def compute_metrics_from_returns(r):
    ann_ret = r.mean() * TRADING_DAYS
    ann_vol = r.std(ddof=1) * np.sqrt(TRADING_DAYS)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    mdd = max_drawdown_from_returns(r)
    return ann_ret, sharpe, mdd


rows = []

for r in boot_paths:
    ann_ret, sharpe, mdd = compute_metrics_from_returns(r)
    rows.append({
        "Annual Return": ann_ret,
        "Sharpe": sharpe,
        "Max Drawdown": mdd
    })

exp01d = pd.DataFrame(rows)
exp01d.head()


exp01d.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])

,Annual Return,Sharpe,Max Drawdown
count,1000.000000,1000.000000,1000.000000
mean,0.089008,0.812618,-0.202592
std,0.033843,0.320118,0.059837
min,-0.024955,-0.208905,-0.461138
5%,0.032984,0.285180,-0.314810
25%,0.066374,0.602662,-0.236031
50%,0.089064,0.803037,-0.191830
75%,0.111860,1.016665,-0.158759
95%,0.143710,1.354933,-0.125512
max,0.200853,1.917667,-0.078249
